<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A4_0426.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install geopandas osmnx networkx rasterio rasterstats shapely

In [35]:
# ===== 라이브러리 =====

import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import networkx as nx
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Point

from google.colab import drive
drive.mount('/content/drive')


BASE = '/content/drive/MyDrive/전종설/0426'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
import geopandas as gpd

# 파일이 있는 경로로 수정
GU_SHP = f'{BASE}/행정구역/행정구역.shp'

gu = gpd.read_file(GU_SHP, encoding='cp949')
print("CRS:", gu.crs)
print("컬럼:", gu.columns.tolist())
print("행 수:", len(gu))
print(gu.head())

CRS: None
컬럼: ['SIDO_CD', 'SIDO_NM', 'SIGUNGU_CD', 'SIGUNGU_NM', 'ADM_CD', 'ADM_NM', 'geometry']
행 수: 424
  SIDO_CD SIDO_NM SIGUNGU_CD SIGUNGU_NM   ADM_CD ADM_NM  \
0      11   서울특별시      11010        종로구  1101053    사직동   
1      11   서울특별시      11010        종로구  1101054    삼청동   
2      11   서울특별시      11010        종로구  1101055    부암동   
3      11   서울특별시      11010        종로구  1101056    평창동   
4      11   서울특별시      11010        종로구  1101057    무악동   

                                            geometry  
0  POLYGON ((953553.932 1953335.741, 953555.211 1...  
1  POLYGON ((953844.081 1955492.177, 953858.644 1...  
2  POLYGON ((952560.239 1956394.672, 952554.005 1...  
3  POLYGON ((953683.828 1959209.872, 953647.333 1...  
4  POLYGON ((952386.354 1952848.711, 952381.494 1...  


In [37]:
# ===== STEP 1: 4개 DEM 합치기 =====
DEM_SEOUL_IMG  = f'{BASE}/서울dem.img'    # 37608
DEM_ANYANG_IMG = f'{BASE}/안양dem.img'    # 37612
DEM_SEONGDONG_IMG = f'{BASE}/성동dem.img' # 37705
DEM_SUWON_IMG  = f'{BASE}/수원dem.img'    # 37709
DEM_MERGED = f'{BASE}/dem_merged.tif'

with rasterio.open(DEM_SEOUL_IMG) as src1, \
     rasterio.open(DEM_ANYANG_IMG) as src2, \
     rasterio.open(DEM_SEONGDONG_IMG) as src3, \
     rasterio.open(DEM_SUWON_IMG) as src4:
    mosaic, out_transform = merge([src1, src2, src3, src4])
    out_profile = src1.profile.copy()
    out_profile.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform
    })

with rasterio.open(DEM_MERGED, 'w', **out_profile) as dst:
    dst.write(mosaic)

# 범위 확인 — x가 ~970000까지 늘어나야 정상
with rasterio.open(DEM_MERGED) as src:
    print("합친 DEM 범위:", src.bounds)

합친 DEM 범위: BoundingBox(left=933390.0, bottom=1916190.0, right=978390.0, top=1972440.0)


In [38]:
# ===== STEP 2: 경사도 래스터 생성 =====
with rasterio.open(DEM_MERGED) as src:
    dem = src.read(1).astype(float)
    transform = src.transform
    res_x = transform[0]
    res_y = -transform[4]
    nodata = src.nodata
    profile = src.profile.copy()

if nodata is not None:
    dem[dem == nodata] = np.nan

# 기존 코드와 동일한 경사도 계산
dy, dx = np.gradient(dem, res_y, res_x)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

print(f"경사도 범위: {np.nanmin(slope_deg):.1f}° ~ {np.nanmax(slope_deg):.1f}°")
print(f"경사도 평균: {np.nanmean(slope_deg):.1f}°")

경사도 범위: 0.0° ~ 52.8°
경사도 평균: 7.0°


In [39]:
# ===== STEP 3: 산악지형 마스킹 =====
# 방법: 경사도 임계값으로 생활권 외 지형 제외
# 15° 이상 = 일반적으로 산지/개발 불가 지형으로 분류

SLOPE_THRESHOLD = 15.0  # 도(°) 단위 — 필요시 조정 가능

slope_masked = slope_deg.copy()
slope_masked[slope_deg >= SLOPE_THRESHOLD] = np.nan  # 산악 픽셀 제거

# (선택) 고도 임계값도 같이 적용하면 더 정확
ELEVATION_THRESHOLD = 100  # 100m 이상 고지대 추가 제외
slope_masked[dem >= ELEVATION_THRESHOLD] = np.nan

remaining_pct = np.sum(~np.isnan(slope_masked)) / np.sum(~np.isnan(slope_deg)) * 100
print(f"✅ 마스킹 후 잔여 픽셀: {remaining_pct:.1f}%")


✅ 마스킹 후 잔여 픽셀: 68.2%


In [40]:
# ===== STEP 4: 경사도 래스터 저장 =====
profile.update(dtype='float32', nodata=np.nan, count=1, driver='GTiff')

with rasterio.open(SLOPE_TIF, 'w', **profile) as dst:
    dst.write(slope_masked.astype('float32'), 1)

print(f"✅ 마스킹된 경사도 저장: {SLOPE_TIF}")

✅ 마스킹된 경사도 저장: /content/drive/MyDrive/전종설/0330/slope_seoul.tif


In [41]:
# ===== STEP 5: 자치구별 평균 경사도 집계 =====
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

# 행정동 경계 로드 후 자치구 단위로 dissolve
gu_raw = gpd.read_file(GU_SHP, encoding='cp949')

# CRS가 None이므로 수동 설정 (좌표값으로 보아 EPSG:5179)
gu_raw = gu_raw.set_crs(epsg=5179)

# 행정동 → 자치구 단위로 합치기
gu = gu_raw.dissolve(by='SIGUNGU_NM').reset_index()
print(f"자치구 수: {len(gu)}개")  # 25가 나와야 정상

# DEM CRS로 맞추기
with rasterio.open(SLOPE_TIF) as src:
    dem_crs = src.crs

gu = gu.to_crs(dem_crs)

results = []

for _, row in gu.iterrows():
    geom = [mapping(row.geometry)]

    try:
        with rasterio.open(SLOPE_TIF) as src:
            out_image, _ = rio_mask(src, geom, crop=True, nodata=np.nan)

        vals = out_image[0]
        valid_vals = vals[~np.isnan(vals)]

        mean_slope = float(np.mean(valid_vals)) if len(valid_vals) > 0 else np.nan

    except Exception as e:
        print(f"⚠️ {row['SIGUNGU_NM']} 오류: {e}")
        mean_slope = np.nan

    results.append({
        '자치구': row['SIGUNGU_NM'],
        '평균경사도(°)': round(mean_slope, 2) if not np.isnan(mean_slope) else None
    })

result_df = pd.DataFrame(results).sort_values('평균경사도(°)', ascending=False)
print(result_df.to_string(index=False))


result_df = pd.DataFrame(results).sort_values('평균경사도(°)', ascending=False)

print("\n=== 서울시 자치구별 평균 경사도 (산악 제외) ===")
print(result_df.to_string(index=False))


자치구 수: 25개
 자치구  평균경사도(°)
서대문구      5.61
 관악구      5.45
 은평구      5.26
 동작구      5.13
 성북구      4.53
 종로구      4.17
 서초구      4.16
 강북구      4.12
  중구      3.73
 용산구      3.54
 구로구      3.49
 도봉구      3.35
 노원구      3.32
 강남구      3.31
 중랑구      3.14
 금천구      3.09
 성동구      2.88
 양천구      2.83
 마포구      2.68
 광진구      2.43
동대문구      2.19
 강동구      2.11
 강서구      1.96
 송파구      1.37
영등포구      1.12

=== 서울시 자치구별 평균 경사도 (산악 제외) ===
 자치구  평균경사도(°)
서대문구      5.61
 관악구      5.45
 은평구      5.26
 동작구      5.13
 성북구      4.53
 종로구      4.17
 서초구      4.16
 강북구      4.12
  중구      3.73
 용산구      3.54
 구로구      3.49
 도봉구      3.35
 노원구      3.32
 강남구      3.31
 중랑구      3.14
 금천구      3.09
 성동구      2.88
 양천구      2.83
 마포구      2.68
 광진구      2.43
동대문구      2.19
 강동구      2.11
 강서구      1.96
 송파구      1.37
영등포구      1.12


In [42]:
# 어떤 자치구가 현재 DEM 범위 밖인지 정확히 파악
with rasterio.open(DEM_MERGED) as src:
    dem_bounds = src.bounds
    print("현재 DEM 범위:")
    print(f"  x: {dem_bounds.left:.0f} ~ {dem_bounds.right:.0f}")
    print(f"  y: {dem_bounds.bottom:.0f} ~ {dem_bounds.top:.0f}")

print("\nNaN 자치구 중심 좌표:")
nan_gus = ['강남구','강동구','광진구','노원구','도봉구','동대문구','성동구','송파구','중랑구']
for _, row in gu[gu['SIGUNGU_NM'].isin(nan_gus)].iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    print(f"  {row['SIGUNGU_NM']}: x={cx:.0f}, y={cy:.0f}")

현재 DEM 범위:
  x: 933390 ~ 978390
  y: 1916190 ~ 1972440

NaN 자치구 중심 좌표:
  강남구: x=961370, y=1944245
  강동구: x=968820, y=1950182
  광진구: x=963407, y=1949790
  노원구: x=962515, y=1961531
  도봉구: x=958760, y=1963390
  동대문구: x=960698, y=1953718
  성동구: x=959461, y=1950286
  송파구: x=965998, y=1945219
  중랑구: x=964062, y=1955456


In [43]:
# ===== STEP 6: (선택) 결과 저장 =====
OUT_CSV = f'{BASE}/자치구별_평균경사도.csv'
result_df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ 결과 저장 완료: {OUT_CSV}")


✅ 결과 저장 완료: /content/drive/MyDrive/전종설/0426/자치구별_평균경사도.csv
